# 11 - Task 3: per-group share, the duplicate leak, and the leaderboard probe

The cheapest thing in round 5 and the first to run. **No model is trained here.** Every
change is to how existing predictions are turned into labels.

**The observation.** The 6,999 test rows are not one population. 1,999 have UUID-style
ids and 5,000 have numeric ids, and measured on the raw text they are plainly different
kinds of document: the numeric group runs to a median 1,862 characters against the UUID
group's 1,189, carries markdown `**` at 2.52 occurrences per document against 0.44, and
contains the word "reviewer" in 15.8% of documents against 0.2%.

The COLING paper explains it. The shared task's training data is HC3 + M4GT + MAGE; the
test split is CUDRT + IELTS + NLPeer + PeerSum + MixSet, with no overlap. NLPeer and
PeerSum are academic peer reviews, which is what the numeric group looks like. The
paper's Table 1 puts the training distribution at 62.6% machine and the test split at
53.1%, matching the 0.6252 this project measured on train and the ~0.53 it inferred for
test.

**So the two groups should not share a predicted class balance, and right now they do.**
Every submission this project has made applies one global share to all 6,999 rows.
Given that share moves the score roughly four times more than model choice does, that is
the largest untested lever available.

**Three things happen in this notebook**, in increasing order of speculativeness:

1. **The duplicate leak.** Ten test texts appear verbatim in train, so their labels are
   known for free. Nothing has noticed them.
2. **Per-group share.** Threshold the two id groups separately.
3. **The leaderboard probe.** One submission that finally settles whether the public
   leaderboard scores the UUID rows or the numeric rows, which has been an open question
   since round 3 and gates the final private-leaderboard pick.

**Honest limitation, up front:** none of this can be validated locally. Notebook 09
established that dev and holdout are both carved from train and inherit its 0.6252, so
they cannot speak to the test set's class balance. The evidence here is Kaggle-only.

## 0. Setup

In [ ]:
# Reload src/ helpers on every cell execution. Round 5 adds src/text.py,
# src/text_features.py and src/clustering.py while these notebooks are open, and a
# plain `import` caches the module in the kernel - so a fixed helper keeps failing
# with the OLD traceback until the kernel is restarted. With this, saving the .py
# is enough.
%load_ext autoreload
%autoreload 2

# Adds project root to path so `import src...` works from notebooks/.
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import paths, data, evaluation, tuning, ensemble, text
from src.paths import FIGURES
FIGURES.mkdir(parents=True, exist_ok=True)

from src import clustering

## 1. Load the raw text and split the test set by id format

`src/text.py` is new in round 5. Nothing before it ever opened `train.csv` or
`test.csv` except one length `describe()` in 01_eda section 3.

The id split is exact rather than heuristic: an id is either all digits or it is not.

In [ ]:
train_ids, train_texts, y = text.load_train_text()
test_ids, test_texts = text.load_test_text()

groups = text.id_group(test_ids)
masks = text.group_masks(test_ids)

print(f"train {len(train_ids)} rows, test {len(test_ids)} rows")
print(f"train id groups: {pd.Series(text.id_group(train_ids)).value_counts().to_dict()}")
print(f"test  id groups: {pd.Series(groups).value_counts().to_dict()}")

assert masks["numeric"].sum() == 5000 and masks["uuid"].sum() == 1999, \
    "id-group counts changed - the whole notebook is built on this split"

## 2. Evidence that the two groups are different populations

If this table came back flat, the rest of the notebook would be unmotivated and should
be abandoned. It is here so that the premise is checked rather than assumed.

In [ ]:
rows = []
for name, sub in [("train", train_texts),
                  ("test uuid", test_texts[masks["uuid"]]),
                  ("test numeric", test_texts[masks["numeric"]])]:
    s = text.text_summary(sub).iloc[0]
    rows.append({"group": name, **s.to_dict()})
evidence = pd.DataFrame(rows).set_index("group")
print(evidence.round(4).to_string())

print("\nPeer-review vocabulary, fraction of documents containing the term:")
for term in ["reviewer", "Strengths", "Weaknesses", "**"]:
    frac = {"train": np.mean([term in t for t in train_texts]),
            "test uuid": np.mean([term in t for t in test_texts[masks["uuid"]]]),
            "test numeric": np.mean([term in t for t in test_texts[masks["numeric"]]])}
    print(f"  {term:12s} " + "  ".join(f"{k} {v:.3f}" for k, v in frac.items()))

print("\nRead this as: the UUID rows look like train, the numeric rows do not.")

## 3. The duplicate leak

Ten texts appear verbatim in both files. Matched on exact text, never on id: the id sets
are disjoint by construction, so an id join finds nothing.

This is small (10 rows out of 6,999, so at most about 0.0014 of macro F1 even if every
one were currently wrong) and it is free. It is included because leaving known labels on
the table is indefensible, not because it will move the leaderboard on its own.

In [ ]:
dups = text.find_train_test_duplicates(train_ids, train_texts, y, test_ids, test_texts)
print(f"{len(dups)} test rows have a verbatim train duplicate")
print(dups.to_string(index=False) if len(dups) else "(none)")

if len(dups):
    print(f"\nlabel distribution among them: {dups['label'].value_counts().to_dict()}")
    print(f"id groups: {dups['group'].value_counts().to_dict()}")

# Guard from the plan: the exploration measured exactly 10, all label 0, 5 per group.
# A different count means the matching logic changed, not that the data did.
if len(dups) != 10:
    print(f"\nWARNING: expected 10 duplicates, found {len(dups)}. "
          "Check the matching logic before trusting the patched submission.")

## 4. Recover the current best test scores

Per-group thresholding needs continuous scores, and the submission CSVs hold only
labels. Rather than refit anything, this reloads the members that
`10_stacked_ensemble.ipynb` section 13 already fitted on all 20,000 rows and re-scores
the test set, then rebuilds the winning blend from the combiner ledger.

If the pickle is missing (it is gitignored), the fallback is to re-run notebook 10
section 13, which is the only expensive step in this notebook.

In [ ]:
from src import combiners

refit_path = paths.MODELS / "round4_members_full_refit.pkl"
assert refit_path.exists(), (
    f"{refit_path.name} not found - it is gitignored, so re-run "
    "10_stacked_ensemble.ipynb section 13 on this machine first")

# joblib.load unpickles, which executes arbitrary code, so it is only ever safe on a
# file you produced yourself. This one is written by 10_stacked_ensemble section 13 on
# this machine and models/ is gitignored, so it is never fetched from anywhere. If the
# file is missing, regenerate it rather than obtaining a copy from a teammate.
fitted = joblib.load(refit_path)
Xt, test_ids_feat = data.load_test_features(sparse=True)
assert list(test_ids_feat) == list(test_ids), "feature and text id order disagree"

test_scores = {}
for name, obj in fitted.items():
    if isinstance(obj, list):          # lightgbm was fitted once per seed
        test_scores[name] = ensemble.seed_average(
            [ensemble.member_score(f, Xt) for f in obj])
    else:
        test_scores[name] = ensemble.member_score(obj, Xt)

members = sorted(test_scores)
R_test, members = ensemble.rank_matrix(test_scores, members)
print(f"re-scored {len(members)} members: {members}")

In [ ]:
# Rebuild the round-4 winner. hill_climb scored 0.73754 on Kaggle against meta/gbm's
# 0.73604, and it carries a ~40x smaller optimism gap, so it is the one to build on.
lanes = combiners.load_results()
assert len(lanes), "no combiner results on disk - git pull the ensemble_trials/"

pick = lanes[(lanes["lane"] == "weights") & (lanes["kind"] == "hill_climb")]
assert len(pick), "the hill_climb result is missing from the ledger"
best_config = pick.iloc[0]["config"]

# The blend must be refitted on the dev OOF matrix, exactly as notebook 10 did, so the
# weights are the same ones that produced the 0.73754 submission.
oof = {m: tuning.load_oof(m) for m in members}
dev_idx = np.load(paths.DATA_PROCESSED / "dev_idx.npy")
y_dev = y[dev_idx]
R_dev, _ = ensemble.rank_matrix(oof, members)

winner = combiners.fit_full("weights", best_config, R_dev, y_dev)
base_scores = winner(R_test)

print(f"config: {best_config}")
print(f"weights: {dict(zip(members, np.round(winner.info['weights'], 3)))}")

# Sanity: reproducing the shipped file exactly confirms the scores are the right ones.
BENCH = "ensemble_weights_hill_climb_share50.csv"
if (paths.SUBMISSIONS / BENCH).exists():
    shipped = pd.read_csv(paths.SUBMISSIONS / BENCH)["label"].to_numpy()
    rebuilt = ensemble.threshold_at_share(base_scores, 0.4996)
    print(f"\nrebuilt vs shipped {BENCH}: {(rebuilt != shipped).sum()} rows differ")
    print("0 differing rows means the recovered scores are exactly the shipped ones.")

## 5. Choosing the two shares

The paper gives true machine shares of 62.5% for the training distribution (which the
UUID rows match) and 53.1% for the test split (the numeric rows).

Those are *true* shares, not the shares to predict. Macro F1 is not maximised by
predicting the true prior, and this project has measured the offset: the leaderboard
peaked at a predicted 0.4996 when the file's blended true share is about
`(0.625 x 1999 + 0.531 x 5000) / 6999 = 0.558`, a ratio of about 0.895. Applying that
same ratio to each group separately gives roughly **0.56 for UUID** and **0.475 for
numeric**.

**That derivation is a heuristic and is labelled as one.** It assumes the offset between
true and optimal predicted share is a constant ratio, which is not something this
project has established. It is used to pick a sensible bracket, not as a point estimate
worth betting a submission on.

Note also what the numbers do: the global predicted share barely moves (0.4996 to about
0.499), so this is a pure *redistribution* between the groups rather than a share change
in disguise. That keeps it separable from everything notebook 09 already settled.

In [ ]:
TRUE_SHARE = {"uuid": 0.625, "numeric": 0.531}   # COLING paper, Table 1
BENCH_SHARE = 0.4996                              # what every current submission uses

blended_true = sum(TRUE_SHARE[g] * masks[g].sum() for g in masks) / len(test_ids)
offset = BENCH_SHARE / blended_true
target = {g: round(TRUE_SHARE[g] * offset, 4) for g in masks}

print(f"blended true share      {blended_true:.4f}")
print(f"offset (0.4996 / true)  {offset:.4f}")
print(f"per-group targets       {target}")

implied = sum(target[g] * masks[g].sum() for g in masks) / len(test_ids)
print(f"\nimplied global share    {implied:.4f}  (benchmark {BENCH_SHARE})")
print("Close to the benchmark by construction: this redistributes share between the "
      "groups rather than changing it overall.")

## 6. The leaderboard probe

This has been an open question since round 3: does the public leaderboard
score the 1,999 UUID rows, the 5,000 numeric rows, or a random mix? It matters because
the two final private-leaderboard picks should hedge differently in each case, and
because it determines whether public feedback says anything at all about the private
rows.

**The probe.** Submit a file identical to the current best *except* that only the UUID
rows are re-thresholded. The numeric rows keep their existing labels exactly.

| public score | conclusion |
|---|---|
| unchanged from 0.73754 | the public set contains no UUID rows, so public = numeric |
| moves | the public set contains UUID rows |

One submission, and the answer is unambiguous either way. It also happens to be a
plausible improvement rather than a pure diagnostic, so the slot is not wasted.

In [ ]:
def patch_duplicates(preds, dup_frame, ids):
    '''Force known-label rows to their known label. Returns (preds, n_changed).'''
    if not len(dup_frame):
        return preds, 0
    preds = preds.copy()
    pos = pd.Series(np.arange(len(ids)), index=ids)
    idx = pos.loc[dup_frame["test_id"]].to_numpy()
    changed = int((preds[idx] != dup_frame["label"].to_numpy()).sum())
    preds[idx] = dup_frame["label"].to_numpy()
    return preds, changed


baseline = ensemble.threshold_at_share(base_scores, BENCH_SHARE)

# A: probe. Only the UUID rows move; numeric rows keep the benchmark cut.
probe_targets = {"uuid": target["uuid"], "numeric": BENCH_SHARE}
probe = clustering.threshold_per_group(base_scores, groups, probe_targets)
probe, n_probe_dup = patch_duplicates(probe, dups, test_ids)

# B: both groups at their own target.
both = clustering.threshold_per_group(base_scores, groups, target)
both, n_both_dup = patch_duplicates(both, dups, test_ids)

for name, preds, extra in [("probe (uuid only)", probe, n_probe_dup),
                           ("per-group (both)", both, n_both_dup)]:
    diff_u = int((preds[masks["uuid"]] != baseline[masks["uuid"]]).sum())
    diff_n = int((preds[masks["numeric"]] != baseline[masks["numeric"]]).sum())
    print(f"{name:20s} share {preds.mean():.4f}  "
          f"changed: {diff_u} uuid, {diff_n} numeric  ({extra} from duplicates)")

print("\nFor the probe, 'changed numeric' should be 0 except for duplicate patches - "
      "that is what makes the public-score comparison clean.")

## 7. Write the batch and run the guards

In [ ]:
sample = pd.read_csv(paths.DATA_RAW / "sample_submission.csv", dtype={"id": str})
assert list(test_ids) == list(sample["id"]), "test ids do not match sample_submission"

batch = {"probe_uuid_share56.csv": probe,
         "pergroup_share56_48.csv": both}

for fname, preds in batch.items():
    data.write_submission(test_ids, preds, fname)

rows = []
for fname, preds in batch.items():
    sub = pd.read_csv(paths.SUBMISSIONS / fname, dtype={"id": str})
    assert list(sub.columns) == ["id", "label"], sub.columns
    assert list(sub["id"]) == list(sample["id"]), f"{fname}: id order drifted"
    assert set(sub["label"].unique()) <= {0, 1}, f"{fname}: labels outside 0/1"
    lab = sub["label"].to_numpy()
    for g, m in masks.items():
        # Per-group realised share must hit its target to within one row.
        want = probe_targets[g] if fname.startswith("probe") else target[g]
        assert abs(lab[m].mean() - want) < 1 / m.sum(), \
            f"{fname}: {g} share {lab[m].mean():.4f} != {want}"
    rows.append({"file": fname, "share": round(float(lab.mean()), 4),
                 "uuid_share": round(float(lab[masks['uuid']].mean()), 4),
                 "numeric_share": round(float(lab[masks['numeric']].mean()), 4),
                 "rows_vs_best": int((lab != baseline).sum())})
    print(f"{fname:28s} OK")

batch_table = pd.DataFrame(rows)
print()
print(batch_table.to_string(index=False))

ledger = paths.DATA_PROCESSED / "round5_results.csv"
out = batch_table.copy()
out["kaggle_f1"] = np.nan
if ledger.exists():
    prev = pd.read_csv(ledger)[["file", "kaggle_f1"]].dropna()
    if len(prev):
        out = out.drop(columns="kaggle_f1").merge(prev, on="file", how="left")
out.to_csv(ledger, index=False)
print(f"\nledger: {ledger}")
print("Benchmark: ensemble_weights_hill_climb_share50.csv = 0.73754")
print("Noise floor 0.0084. Anything inside that is a tie.")

## Discussion / carry-forward -> `12_text_features.ipynb`

_Fill in once the two files are scored._

**Reading the probe.** Compare `probe_uuid_share56.csv` against 0.73754. Only UUID rows
changed, so:

- **Score identical** (to 5 decimal places): the public leaderboard contains no UUID
  rows. Public = the numeric subset, private = the UUID rows or a split of them. This
  would mean every public score this project has ever read describes the peer-review
  domain only, and the final picks should be chosen for that domain.
- **Score moves**: the public set includes UUID rows. Record the size of the move, since
  it also bounds what fraction of the public set they make up.

**Reading the per-group file.** `pergroup_share56_48.csv` against 0.73754 measures
whether splitting the share helps at all. Remember the caveat from section 5: the 0.895
offset is a heuristic. If this file moves the score by more than the 0.0084 noise floor
in either direction, the per-group idea is worth a second submission to bracket the
targets. If it lands inside the noise, stop here rather than spending more slots.

**What this cannot tell you.** Nothing here is locally validated, by construction. Do
not let a good Kaggle result from this notebook be read as evidence about the model.

**Carry forward:** the id-group split is now available as `text.group_masks`, and
notebook 13 will check whether unsupervised clustering recovers it without being told.